# Data Visualization: Pola Pemilihan Mata Kuliah Mahasiswa
**Mata Kuliah:** Data Mining (Pertemuan 6)  
**Nama / NIM:** *(isi nama dan NIM kamu)*  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Plotly

## Tujuan Analisis
Mengeksplorasi data pilihan mata kuliah dari 100 mahasiswa untuk menjawab:
1. Mata kuliah apa yang paling dan paling sedikit diminati?
2. Berapa banyak mata kuliah yang umumnya dipilih mahasiswa (beban pilihan)?
3. Rumpun keilmuan mana yang dominan?
4. Apakah ada mata kuliah yang cenderung dipilih bersamaan?
5. Bagaimana profil pilihan berbeda antar kategori beban (Ringan, Sedang, Tinggi)?

## 1. Import Library dan Load Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
os.makedirs("images", exist_ok=True)

def simpan(nama):
    plt.savefig(f"images/{nama}.png", dpi=150, bbox_inches="tight")

# Di Google Colab: upload clean_data.csv jika file belum ada
if not os.path.exists("clean_data.csv"):
    from google.colab import files
    files.upload()

df = pd.read_csv("clean_data.csv")
df.head()

## 2. Data Understanding

In [ ]:
print("Ukuran data (baris, kolom):", df.shape)
df.info()

In [ ]:
print("Missing value per kolom:")
print(df.isnull().sum())
print("\nData duplikat:", df.duplicated().sum())
df.describe().T

**Deskripsi dataset.** Setiap baris adalah satu mahasiswa. Sembilan kolom mata kuliah bernilai biner (1 = dipilih, 0 = tidak dipilih). Kolom turunan: `jumlah_pilihan` (total mata kuliah yang dipilih), `kategori_beban` (Ringan 1-3, Sedang 4-5, Tinggi 6-9), dan empat kolom `n_...` yang menghitung pilihan per rumpun keilmuan. Data sudah bersih: tidak ada missing value.

## 3. Data Preparation
Kolom identitas (`Nama`, `NIM`) dihapus agar analisis dan repository publik tidak memuat data pribadi.

In [ ]:
df = df.drop(columns=[c for c in ["Nama", "NIM"] if c in df.columns])

matkul = ["Data mining", "Desain Grafis", "Sistem Informasi Pendidikan",
          "Teknologi IoT", "Pengolahan Citra Digital", "Pemrograman CMS",
          "Realitas Virtual", "Game Edukasi", "Sistem Keamanan Jaringan"]
rumpun = ["n_Data dan Sistem Informasi", "n_Jaringan dan IoT",
          "n_Multimedia dan Visual", "n_Pengembangan Web"]
urutan_beban = ["Ringan (1-3)", "Sedang (4-5)", "Tinggi (6-9)"]

# Validasi konsistensi kolom turunan
print("jumlah_pilihan konsisten:", (df[matkul].sum(axis=1) == df["jumlah_pilihan"]).all())
print("Total rumpun konsisten  :", (df[rumpun].sum(axis=1) == df["jumlah_pilihan"]).all())

## 4. Visualisasi dan Interpretasi

### 4.1 Bar Chart: Popularitas Setiap Mata Kuliah

In [ ]:
pop = (df[matkul].mean() * 100).sort_values()

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=pop.values, y=pop.index, color="#3B7DD8")
for i, v in enumerate(pop.values):
    ax.text(v + 1, i, f"{v:.0f}%", va="center")
plt.title("Popularitas Mata Kuliah (% dari 100 Mahasiswa)")
plt.xlabel("Persentase mahasiswa yang memilih (%)")
plt.ylabel("Mata kuliah")
plt.xlim(0, 105)
simpan("01_popularitas_matkul")
plt.show()

**Interpretasi.** Pemrograman CMS paling diminati (94%), disusul Realitas Virtual (87%) dan Pengolahan Citra Digital (68%). Data Mining (64%) dan Game Edukasi (62%) berada di tengah. Sebaliknya, Sistem Informasi Pendidikan hanya dipilih 10% mahasiswa, dan Sistem Keamanan Jaringan 24%. Selisih antara yang terpopuler dan terendah sangat besar (84 poin persentase), sehingga minat mahasiswa tidak merata. Mata kuliah praktis berbasis pengembangan dan visual lebih menarik dibanding yang berorientasi sistem informasi dan keamanan.

### 4.2 Histogram: Distribusi Jumlah Mata Kuliah yang Dipilih

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["jumlah_pilihan"], bins=np.arange(0.5, 9.5, 1), color="#3B7DD8")
plt.axvline(df["jumlah_pilihan"].mean(), color="crimson", linestyle="--",
            label=f"Rata-rata = {df['jumlah_pilihan'].mean():.2f}")
plt.axvline(df["jumlah_pilihan"].median(), color="black", linestyle=":",
            label=f"Median = {df['jumlah_pilihan'].median():.0f}")
plt.title("Distribusi Jumlah Mata Kuliah yang Dipilih per Mahasiswa")
plt.xlabel("Jumlah mata kuliah dipilih")
plt.ylabel("Jumlah mahasiswa")
plt.legend()
simpan("02_histogram_jumlah_pilihan")
plt.show()

**Interpretasi.** Distribusi hampir simetris dan terpusat pada 4-5 mata kuliah (rata-rata 4,79; median dan modus 5). Sebanyak 39 mahasiswa memilih 5 mata kuliah dan 28 memilih 4. Nilai ekstrem jarang terjadi: hanya 1 mahasiswa memilih satu mata kuliah, dan 2 mahasiswa memilih delapan. Kebanyakan mahasiswa memilih beban menengah, bukan minimal maupun maksimal.

### 4.3 Bar Chart: Komposisi Kategori Beban

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="kategori_beban", order=urutan_beban,
                   hue="kategori_beban", palette="Blues", legend=False)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} ({p.get_height()/len(df)*100:.0f}%)",
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.title("Jumlah Mahasiswa per Kategori Beban")
plt.xlabel("Kategori beban")
plt.ylabel("Jumlah mahasiswa")
plt.ylim(0, 80)
simpan("03_kategori_beban")
plt.show()

**Interpretasi.** Kategori Sedang (4-5 mata kuliah) mendominasi dengan 67 mahasiswa (67%), diikuti Tinggi dengan 23 mahasiswa (23%) dan Ringan dengan 10 mahasiswa (10%). Pola ini konsisten dengan histogram sebelumnya: mayoritas mahasiswa mengambil beban moderat. Kelompok Ringan paling kecil, sehingga perbandingan yang melibatkan kelompok ini perlu dibaca hati-hati karena jumlah sampelnya sedikit.

### 4.4 Bar Chart: Total Pilihan per Rumpun Keilmuan

In [ ]:
tot = df[rumpun].sum().rename(lambda s: s.replace("n_", ""))
jml_mk = pd.Series([2, 2, 4, 1], index=tot.index)  # jumlah mata kuliah per rumpun
tabel = pd.DataFrame({"Total pilihan": tot, "Jumlah mata kuliah": jml_mk,
                      "Rata-rata pilihan per mata kuliah": tot / jml_mk})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(x=tabel["Total pilihan"], y=tabel.index, color="#3B7DD8", ax=axes[0])
axes[0].set_title("Total Pilihan per Rumpun")
axes[0].set_xlabel("Total pilihan (seluruh mahasiswa)")
axes[0].set_ylabel("Rumpun keilmuan")
sns.barplot(x=tabel["Rata-rata pilihan per mata kuliah"], y=tabel.index, color="#E8913A", ax=axes[1])
axes[1].set_title("Rata-rata Pilihan per Mata Kuliah dalam Rumpun")
axes[1].set_xlabel("Rata-rata pilihan per mata kuliah")
axes[1].set_ylabel("")
plt.tight_layout()
simpan("04_rumpun_keilmuan")
plt.show()
tabel

**Interpretasi.** Rumpun Multimedia dan Visual unggul dalam total pilihan (249), tetapi rumpun ini memuat empat mata kuliah. Setelah dinormalisasi per mata kuliah, Pengembangan Web (satu mata kuliah, 94 pilihan) justru yang paling kuat, disusul Multimedia dan Visual (rata-rata sekitar 62), lalu Data dan Sistem Informasi (37) dan Jaringan dan IoT (31). Jadi total mentah dapat menyesatkan: dominasi Multimedia sebagian berasal dari banyaknya mata kuliah di rumpun tersebut, bukan hanya dari minat.

### 4.5 Boxplot: Sebaran Jumlah Pilihan per Rumpun

In [ ]:
long = df[rumpun].rename(columns=lambda s: s.replace("n_", "")).melt(
    var_name="Rumpun", value_name="Jumlah pilihan")

plt.figure(figsize=(9, 5))
sns.boxplot(data=long, x="Rumpun", y="Jumlah pilihan", hue="Rumpun",
            palette="Set2", legend=False)
plt.title("Sebaran Jumlah Pilihan per Mahasiswa pada Tiap Rumpun")
plt.xlabel("Rumpun keilmuan")
plt.ylabel("Jumlah mata kuliah dipilih per mahasiswa")
plt.xticks(rotation=15)
simpan("05_boxplot_rumpun")
plt.show()

**Interpretasi.** Multimedia dan Visual memiliki median tertinggi (3 mata kuliah per mahasiswa) dengan rentang 0-4, artinya mahasiswa umumnya mengambil beberapa mata kuliah sekaligus dari rumpun ini. Pengembangan Web hanya bernilai 0 atau 1 karena hanya berisi satu mata kuliah, sehingga hampir seluruh mahasiswa berada di nilai 1. Data dan Sistem Informasi serta Jaringan dan IoT bermedian 1 dengan maksimum 2, menandakan sebagian besar mahasiswa mengambil paling banyak satu mata kuliah dari dua rumpun tersebut.

### 4.6 Heatmap: Korelasi Antar Mata Kuliah

In [ ]:
korelasi = df[matkul].corr()
mask = np.triu(np.ones_like(korelasi, dtype=bool))

plt.figure(figsize=(9, 7))
sns.heatmap(korelasi, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"label": "Korelasi (phi)"})
plt.title("Heatmap Korelasi Antar Mata Kuliah")
plt.grid(False)
simpan("06_heatmap_korelasi")
plt.show()

pasangan = korelasi.where(~mask).stack().dropna().sort_values()
print("Korelasi negatif terkuat:\n", pasangan.head(3).round(2))
print("\nKorelasi positif terkuat:\n", pasangan.tail(3).round(2))

**Interpretasi.** Seluruh 36 pasangan mata kuliah memiliki korelasi lemah (|r| < 0,3). Korelasi positif tertinggi hanya 0,25 (Data Mining dengan Pemrograman CMS), dan korelasi negatif terendah -0,20 (Data Mining dengan Desain Grafis). Artinya tidak ada "paket" mata kuliah yang kuat: memilih satu mata kuliah hampir tidak memprediksi pilihan mata kuliah lain. Karena variabelnya biner, korelasi ini adalah koefisien phi, dan nilainya bisa kecil juga karena beberapa mata kuliah dipilih hampir semua orang (misalnya CMS 94%).

### 4.7 Heatmap: Jumlah Mahasiswa yang Memilih Dua Mata Kuliah Bersamaan

In [ ]:
co = df[matkul].T.dot(df[matkul])   # co-occurrence

plt.figure(figsize=(10, 7))
sns.heatmap(co, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5,
            cbar_kws={"label": "Jumlah mahasiswa"})
plt.title("Co-occurrence: Mahasiswa yang Memilih Kedua Mata Kuliah")
plt.grid(False)
simpan("07_heatmap_cooccurrence")
plt.show()

atas = co.where(np.triu(np.ones_like(co, dtype=bool), 1)).stack().dropna().sort_values(ascending=False)
atas.head(5)

**Interpretasi.** Pasangan yang paling sering dipilih bersamaan adalah Pemrograman CMS dan Realitas Virtual (82 mahasiswa), diikuti Pengolahan Citra Digital dan Pemrograman CMS (65), serta Data Mining dan Pemrograman CMS (63). Angka tinggi ini sebagian besar mencerminkan popularitas masing-masing mata kuliah, bukan keterkaitan khusus, sejalan dengan korelasi yang lemah pada heatmap sebelumnya. Diagonal heatmap sama dengan popularitas tiap mata kuliah pada grafik 4.1.

### 4.8 Heatmap: Profil Pilihan Mata Kuliah per Kategori Beban

In [ ]:
profil = (df.groupby("kategori_beban")[matkul].mean().loc[urutan_beban].T * 100)

plt.figure(figsize=(8, 6))
sns.heatmap(profil, annot=True, fmt=".0f", cmap="Blues", linewidths=0.5,
            cbar_kws={"label": "% mahasiswa yang memilih"})
plt.title("Persentase Pemilihan Mata Kuliah per Kategori Beban")
plt.xlabel("Kategori beban")
plt.ylabel("Mata kuliah")
plt.grid(False)
simpan("08_profil_per_beban")
plt.show()

**Interpretasi.** Mata kuliah yang diambil kelompok Ringan menunjukkan prioritas utama mahasiswa: Pemrograman CMS dan Realitas Virtual (masing-masing 60%) serta Pengolahan Citra Digital (50%). Seiring naiknya beban, Data Mining melonjak dari 20% (Ringan) ke 60% (Sedang) dan 96% (Tinggi), sehingga mata kuliah ini cenderung menjadi tambahan bagi mahasiswa yang mengambil banyak mata kuliah. Sistem Informasi Pendidikan hampir tidak dipilih kelompok Ringan (0%) dan Sedang (6%), baru naik menjadi 26% di kelompok Tinggi. Pemrograman CMS konsisten tinggi di semua kelompok (60% hingga 100%).

### 4.9 Stacked Bar: Komposisi Rumpun per Kategori Beban

In [ ]:
komposisi = (df.groupby("kategori_beban")[rumpun].mean().loc[urutan_beban]
             .rename(columns=lambda s: s.replace("n_", "")))

ax = komposisi.plot(kind="bar", stacked=True, figsize=(9, 6), colormap="Set2")
plt.title("Rata-rata Jumlah Mata Kuliah per Rumpun pada Tiap Kategori Beban")
plt.xlabel("Kategori beban")
plt.ylabel("Rata-rata mata kuliah dipilih")
plt.xticks(rotation=0)
plt.legend(title="Rumpun", bbox_to_anchor=(1.02, 1), loc="upper left")
simpan("09_komposisi_rumpun_beban")
plt.show()
komposisi.round(2)

**Interpretasi.** Multimedia dan Visual menjadi komponen terbesar pada semua kategori beban (rata-rata 1,40 pada Ringan hingga 3,04 pada Tinggi), sehingga rumpun ini adalah "inti" pilihan mahasiswa. Pertumbuhan relatif terbesar ada pada Data dan Sistem Informasi (sekitar 0,20 menjadi 1,22, kira-kira enam kali lipat) dan Jaringan dan IoT (0,40 menjadi 1,09), konsisten dengan pola Data Mining pada heatmap sebelumnya. Pengembangan Web sudah jenuh sejak kategori Sedang (mendekati 1), sehingga tambahan mata kuliah pada beban tinggi datang dari rumpun lain.

### 4.10 Visualisasi Interaktif (Plotly)

In [ ]:
interaktif = profil.reset_index().melt(id_vars="index", var_name="Kategori beban",
                                       value_name="Persentase")
interaktif = interaktif.rename(columns={"index": "Mata kuliah"})

fig = px.bar(interaktif, x="Mata kuliah", y="Persentase", color="Kategori beban",
             barmode="group", category_orders={"Kategori beban": urutan_beban},
             title="Persentase Pemilihan Mata Kuliah per Kategori Beban (Interaktif)",
             labels={"Persentase": "% mahasiswa yang memilih"})
fig.update_layout(xaxis_tickangle=-30)
fig.show()

**Interpretasi.** Grafik interaktif ini memungkinkan perbandingan langsung antar kategori beban per mata kuliah (arahkan kursor untuk melihat nilai, klik legenda untuk menyembunyikan kategori). Pola yang sama terlihat jelas: batang kelompok Tinggi selalu paling tinggi, dan jarak antar kategori paling lebar pada Data Mining, Game Edukasi, dan Teknologi IoT.

> **Catatan:** dataset ini tidak memiliki kolom waktu, sehingga *line chart* tren waktu tidak relevan dan tidak dibuat. Teknik yang dipakai disesuaikan dengan karakteristik data: bar chart, histogram, boxplot, heatmap, stacked bar, dan visualisasi interaktif.

## 5. Kesimpulan
1. **Minat tidak merata.** Pemrograman CMS (94%) dan Realitas Virtual (87%) sangat dominan, sementara Sistem Informasi Pendidikan (10%) dan Sistem Keamanan Jaringan (24%) paling sedikit diminati.
2. **Beban pilihan terpusat di tengah.** Rata-rata mahasiswa memilih 4,79 mata kuliah; 67% berada di kategori Sedang (4-5 mata kuliah).
3. **Multimedia dan Visual adalah rumpun terbesar** (249 pilihan), tetapi setelah dinormalisasi per mata kuliah, Pengembangan Web paling kuat.
4. **Pilihan antar mata kuliah hampir independen.** Seluruh korelasi lemah (|r| < 0,3); pasangan sering muncul bersama terutama karena sama-sama populer.
5. **Mata kuliah tambahan menandai beban tinggi.** Data Mining naik dari 20% (Ringan) menjadi 96% (Tinggi), sehingga berfungsi sebagai pilihan tambahan bagi mahasiswa dengan beban besar.

**Implikasi:** program studi dapat menyiapkan kapasitas kelas untuk mata kuliah populer, mengevaluasi daya tarik mata kuliah dengan peminat rendah, dan menjadikan pola beban sebagai dasar bimbingan akademik.

**Keterbatasan:** data hanya memuat 100 responden, bersifat biner (dipilih/tidak) tanpa informasi alasan, dan kelompok Ringan hanya berisi 10 mahasiswa.